<a href="https://colab.research.google.com/github/bsheese/cs377/blob/main/06_pandas_intro/06_1_Series.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 06.1 — Pandas Series

## Learning Objectives

By the end of this notebook you should be able to:

- Pull a column out of a real dataset and recognize it as a `Series`
- Explain what the **index** is and why it matters
- Access one element or a slice using `.loc[]` and `.iloc[]`
- Apply arithmetic to an entire Series without writing a loop
- Filter a Series down to rows that meet a condition (boolean indexing)
- Use common methods — `.describe()`, `.value_counts()`, `.sort_values()`, `.isnull()` — to summarize and explore data
- Use the `.str` accessor to work with text columns

In [ ]:
import pandas as pd

## The Dataset

We will use data from the Titanic disaster of 1912 throughout this notebook series. The dataset records 887 passengers — their name, sex, age, ticket class, fare paid, and whether they survived.

Our long-term goal is to understand patterns in survival: Who made it? Who didn't? Why? We'll build toward that over six notebooks. For now, we need to master the basic unit of pandas — the **Series** — and the best way to learn it is to use it on real data right away.

In [ ]:
url = "https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_titanic.csv"
df = pd.read_csv(url)
df.columns = df.columns.str.lower()
df = df.rename(columns={
    'siblings/spouses aboard': 'sibsp',
    'parents/children aboard': 'parch'
})
df.head()

You're looking at a **DataFrame** — a table with rows and columns. We'll study DataFrames in depth in the next notebook. For now, notice that each *column* is a list of values of the same type: a column of ages, a column of fares, a column of 0s and 1s for survived.

Each column is actually a pandas **Series**. Let's pull one out.

In [ ]:
ages = df["age"]
ages.head(10)

## 1. What Is a Series?

`ages` is a `Series` — pandas' one-dimensional data structure. It looks a lot like a Python list, but notice the left-hand column of numbers (0, 1, 2, …). That column is the **index**: a label attached to every value.

With a plain Python list, `my_list[3]` gives you the fourth element — but you have no idea what it *represents*. With a Series, every value carries its index label with it, so when you pull out a single entry you still know where it came from.

In [ ]:
print(type(ages))
print()
print("Index:  ", ages.index)
print("Values: ", ages.values[:5], "...")
print("dtype:  ", ages.dtype)

The index here is just row numbers, which is fine for this dataset. But the index can be anything: names, dates, city codes. Notebooks later in this series will show you when a meaningful index is valuable.

## 2. Creating a Series by Hand

In practice you will almost always get a Series by extracting a column from a DataFrame (as above). But knowing how to build one by hand is useful for quick experiments and for understanding the mechanics.

In [ ]:
# From a Python list — pandas assigns a 0-based integer index automatically
sample_ages = pd.Series([22, 38, 26, 35, 28])
sample_ages

In [ ]:
# Supply your own index to make each value self-labeling
sample_ages = pd.Series(
    [22, 38, 26, 35, 28],
    index=["Alice", "Bob", "Carol", "Dan", "Eve"]
)
sample_ages

In [ ]:
# From a dictionary: keys become the index, values become the data
subject_scores = pd.Series({
    "math":    91,
    "english": 84,
    "science": 78,
    "history": 95,
})
subject_scores

## 3. Looking Up a Specific Value

Suppose you want to check one particular passenger's age — maybe you noticed something odd in the data and want to verify it. You need to *look up* a single element.

Pandas gives you two lookup tools, and it's important to know the difference:

| Syntax | Looks up by | Use when |
|---|---|---|
| `s.loc[label]` | **label** (the index value) | your index has meaningful names, or when you want to be explicit |
| `s.iloc[position]` | **integer position** (0, 1, 2 …) | you want the nth element regardless of what the index says |

When the index is just 0, 1, 2 … (as it is for our `ages` column), both give the same result. The difference matters when the index has been set to something else, like names or dates.

In [ ]:
# Look up by label — the label here happens to be the row number
print("Passenger at index 0:", ages.loc[0])
print("Passenger at index 5:", ages.loc[5])

In [ ]:
# Look up by position — always 0-based, regardless of the index
print("First passenger: ", ages.iloc[0])
print("Last passenger:  ", ages.iloc[-1])

In [ ]:
# Slicing a range of rows
# .loc[] end is INCLUSIVE; .iloc[] end is EXCLUSIVE (like Python list slicing)
print("Rows 0 through 4 via .iloc[]:")
print(ages.iloc[0:5])
print()
print("Rows 0 through 4 via .loc[]:")
print(ages.loc[0:4])

## 4. Doing Math on a Whole Column — Without a Loop

Here's something that will save you a lot of time. Suppose you want to convert all fares from British pounds to US dollars (using an approximate rate of £1 = $1.27).

With a Python list, you'd write a loop:
```python
fares_usd = []
for fare in fares_list:
    fares_usd.append(fare * 1.27)
```

With a Series, you write one expression and pandas applies the operation to every element automatically. This is called a **vectorized operation**.

In [ ]:
fares = df["fare"]

# Multiply every fare by 1.27 — no loop
fares_usd = fares * 1.27
fares_usd.head()

In [ ]:
# You can also combine two Series arithmetically.
# Pandas lines them up by index and operates element by element.
# Example: what if every passenger got a £5 group discount?
discount = pd.Series([5.0] * len(fares), index=fares.index)
discounted_fare = fares - discount
discounted_fare.head()

Vectorized operations also work for comparisons — and that's where things get really powerful, as you're about to see.

## 5. Filtering: The Most Useful Thing in This Notebook

When you have 887 rows of data, you almost never want to look at all of them at once. You want to focus on a particular group:

- *Show me only the children.*
- *Show me only passengers who paid more than £50.*
- *Show me only passengers in third class.*

This is called **filtering** (or **subsetting**), and it is the operation you will use more than almost any other in data analysis.

Here is how it works. A comparison like `ages < 18` doesn't return a single True or False — it returns a **boolean Series**: a list of True/False values, one per passenger, marking which ones meet the condition.

In [ ]:
# Which passengers were under 18?
is_child = ages < 18
is_child.head(10)

Now pass that boolean Series back into the brackets of `ages`. Pandas keeps only the rows where the value is `True` and throws away the rest.

In [ ]:
child_ages = ages[is_child]
print(f"Number of passengers under 18: {len(child_ages)}")
child_ages.head()

In [ ]:
# You can write both steps on one line — the style you'll see most often
ages[ages < 18].head()

### Combining Conditions

What if you want passengers who were adults but not elderly — say, between 18 and 60? Combine conditions with `&` (AND) and `|` (OR). **Each condition must be in its own parentheses.**

In [ ]:
working_age = ages[(ages >= 18) & (ages <= 60)]
print(f"Working-age passengers (18–60): {len(working_age)}")

In [ ]:
# Who paid less than £10 OR more than £100? (The extremes)
extreme_fares = fares[(fares < 10) | (fares > 100)]
print(f"Passengers with extreme fares: {len(extreme_fares)}")
extreme_fares.head()

> **Common mistake:** Use `&` and `|`, not Python's `and` / `or` keywords. Python's `and`/`or` compare whole objects (truthy vs falsy) — they don't work element-by-element on a Series and will either raise an error or produce wrong results.

## 6. Summarizing What You Found

Now that you can filter down to a group of interest, the natural next step is to *summarize* that group: how many? what's the average? what's the range? pandas provides a short set of methods for exactly this.

In [ ]:
# .describe() gives a full statistical summary in one call
# Let's describe the ages of children on board
child_ages = ages[ages < 18]
child_ages.describe()

In [ ]:
# Individual summary methods: mean, median, min, max, sum
print(f"Children aboard:       {len(child_ages)}")
print(f"Youngest child:        {child_ages.min()}")
print(f"Oldest child (<18):    {child_ages.max()}")
print(f"Average child age:     {child_ages.mean():.1f}")

In [ ]:
# .value_counts() counts how often each distinct value appears.
# Perfect for categorical columns like passenger class.
pclass = df["pclass"]
pclass.value_counts()

The output tells us there were more third-class passengers than first and second class combined — context that matters when we later compare survival rates.

In [ ]:
# .sort_values() returns a new Series sorted by value; the original is unchanged.
# Let's see the ten cheapest fares.
fares.sort_values().head(10)

In [ ]:
# .unique() lists every distinct value once; .nunique() counts them.
print("Distinct classes:", pclass.unique())
print("Number of distinct classes:", pclass.nunique())

## 7. Missing Values

Real data often has gaps. Some passengers' records are incomplete — maybe age wasn't recorded, or a field was left blank. Pandas represents missing values as `NaN` (Not a Number).

Two methods detect them:
- `.isnull()` returns a boolean Series — `True` wherever data is missing.
- `.notna()` is the opposite.

Chaining `.sum()` on a boolean Series counts the `True` values, giving you the number of missing entries. This is a standard first check on any new dataset.

In [ ]:
# Our Titanic dataset is complete — no missing values
print("Missing ages:", ages.isnull().sum())
print("Missing fares:", fares.isnull().sum())

In [ ]:
# To see what missing values look like, here's a small example
heights = pd.Series(
    [175.0, None, 162.0, None, 180.0],
    index=[0, 1, 2, 3, 4]
)

print(heights)
print()
print("Missing:", heights.isnull().sum())
print()
print("Non-missing heights:")
print(heights[heights.notna()])

Missing values will matter a lot in later notebooks — many Titanic sources have ~20% of ages missing. We'll cover strategies for handling them in the Data Cleaning notebook (06.4).

## 8. Working with Text: the `.str` Accessor

The `name` column contains each passenger's full name, including a title — Mr., Mrs., Miss., Master., Dr., and so on. Titles encode useful information about a passenger's social status and likely sex, which were both factors in survival.

When a Series holds strings, the `.str` accessor unlocks string methods that work element-by-element — no loop needed.

In [ ]:
names = df["name"]
names.head(10)

In [ ]:
# Names look like: "Mr. Owen Harris Braund"
# Extract the title: find the first word that ends with a period
titles = names.str.extract(r'^(\w+\.)')
titles.value_counts()

The four main titles — Mr., Mrs., Miss., Master. — capture almost everyone. `Master.` was the conventional title for boys under roughly 14. This single column therefore encodes both sex and approximate age group.

Other useful `.str` operations:

In [ ]:
# Check whether a name contains a specific string
has_miss = names.str.contains("Miss\.")
print(f"Passengers titled Miss.: {has_miss.sum()}")

In [ ]:
# String methods can be chained
# Normalize to lowercase, strip whitespace — a common cleaning step
messy = pd.Series(["  Mr. Smith ", "MRS. Jones", "miss. lee"])
print(messy.str.strip().str.lower())

## Your Turn

Use the `df["fare"]` column for the following. Each question is designed to practice a different skill from this notebook.

1. **Lookup:** What fare did the passenger at index position 10 pay? Use `.iloc[]`.
2. **Vectorized operation:** Add a 10% tax to every fare. What is the new maximum fare?
3. **Filtering:** How many passengers paid more than £50? What fraction of all passengers is that?
   *(Hint: `.sum()` counts True values; `.mean()` gives the fraction.)*
4. **Combined filter:** How many passengers paid between £10 and £30 (inclusive)?
5. **Summarize:** Among passengers who paid more than £50, what was the average fare? The minimum?
6. **Value counts:** The `df["sex"]` column contains `"male"` and `"female"`. Use `.value_counts()` to see the split. Then use a boolean filter to count how many female passengers there were.

In [ ]:
fare_series = df["fare"]

# Your code here